# Reachy inverse kinematics: go to an xyz point
Frame: origin in the torso, **x forward, y to Reachy's left, z up**, units = metres.
Set `TARGET_XYZ`, run the cell. The hand keeps its current orientation.

In [1]:
from reachy_sdk import ReachySDK
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode
from findIP import get_reachy_ip
import numpy as np, time

np.set_printoptions(precision=3, suppress=True)
reachy = ReachySDK(host='localhost')

ARM_NAME = 'l_arm'
ARM = reachy.l_arm
CHAIN = list(ARM.kinematics_chain.values())   # the 7 arm joints (gripper excluded)
print([j.name for j in CHAIN])

['l_shoulder_pitch', 'l_shoulder_roll', 'l_arm_yaw', 'l_elbow_pitch', 'l_forearm_yaw', 'l_wrist_pitch', 'l_wrist_roll']


In [2]:
#MAX_JOINT_JUMP = 60   # refuse any move that swings a joint more than this (deg)

def current_q():
    return [j.present_position for j in CHAIN]

def move_to(q_target, duration=2.0):
    goto({j: a for j, a in zip(CHAIN, q_target)}, duration=duration,
         interpolation_mode=InterpolationMode.MINIMUM_JERK)

def go_to_xyz(xyz, duration=3.0):
    """IK to xyz (m) with the current hand orientation, check it, then move."""
    q_start = current_q()
    T_goal = ARM.forward_kinematics(joints_position=q_start)   # borrow current orientation
    T_goal[:3, 3] = xyz
    try:
        q_goal = ARM.inverse_kinematics(T_goal, q0=q_start)
    except ValueError as e:
        print('IK failed (probably out of reach):', e)
        return
    err_mm = np.linalg.norm(ARM.forward_kinematics(joints_position=q_goal)[:3, 3] - xyz) * 1000
    jump = np.abs(np.array(q_goal) - np.array(q_start))
    print('joints (deg):', np.round(q_goal, 1))
    # if err_mm > 10:
    #     print(f'IK only gets within {err_mm:.1f} mm: not moving'); return
    # if jump.max() > MAX_JOINT_JUMP:
    #     print(f'joint swing of {jump.max():.0f} deg: not moving'); return
    reachy.turn_on(ARM_NAME)
    move_to(q_goal, duration)
    time.sleep(0.5)
    reached = ARM.forward_kinematics()[:3, 3]
    print('reached (m):', reached, f' error {np.linalg.norm(reached - xyz) * 1000:.1f} mm')

In [9]:
TARGET_XYZ = np.array([.2, 0, -.2])   # metres: x forward, y left, z up
go_to_xyz(TARGET_XYZ)

joints (deg): [  -4.8  -23.3  -22.6  -97.7 -123.9   24.  -165. ]
reached (m): [ 0.334  0.063 -0.288]  error 172.6 mm


In [10]:
# Done: let the arm down gently
reachy.turn_off_smoothly(ARM_NAME)